# verify_fixes.ipynb
Smoke-scale verification of the PROJECT_REVIEW §15/§16 fixes. Run top to bottom on the repo `.venv` kernel (Python 3.9). Each numbered section ends in an assert or execution-check cell. Uses ONLY libraries already installed in `.venv` — no UMAP.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root on path
import numpy as np, torch
print('python', sys.version.split()[0])

python 3.9.6


## 1. metrics_at_threshold uses >= (score==threshold is positive)

In [2]:
from utils.classification_eval import metrics_at_threshold
m, cm = metrics_at_threshold(np.array([1,0]), np.array([0.5,0.2]), 0.5)
assert cm['tp'] == 1 and cm['fn'] == 0, cm
print('OK: score==threshold classified positive')

OK: score==threshold classified positive


## 2. DDPM latent key guard requires opt-in for non-concat_mu

In [3]:
import pytest
from utils.latent_extraction import _validate_ddpm_latent_key, _RECOMMENDED_LATENT_KEYS
_validate_ddpm_latent_key('concat_mu', _RECOMMENDED_LATENT_KEYS, False)
with pytest.raises(ValueError):
    _validate_ddpm_latent_key('fused_cls_mu', _RECOMMENDED_LATENT_KEYS, False)
_validate_ddpm_latent_key('fused_cls_mu', _RECOMMENDED_LATENT_KEYS, True)
print('OK: concat_mu default; fused_cls_mu needs allow_non_concat_mu=True')

OK: concat_mu default; fused_cls_mu needs allow_non_concat_mu=True


## 3. run_gvae.py defaults to latent_quality

In [4]:
src = open('../run_gvae.py').read()
assert '"checkpoint_metric": "latent_quality"' in src
assert 'auc_pr_balanced_accuracy' not in src
print('OK: run_gvae.py defaults to latent_quality')

OK: run_gvae.py defaults to latent_quality


## 4a. Diagnostics carry the threshold-optimism caveat

In [5]:
from utils.classification_eval import build_binary_classification_diagnostics
d = build_binary_classification_diagnostics(np.array([0,0,1,1,0,1]), np.array([.2,.3,.7,.8,.4,.6]))
assert 'threshold_metric_caveat' in d
print(d['threshold_metric_caveat'])

best-threshold F1/balanced_accuracy are selected on the SAME split they are reported on and are therefore optimistic; prefer roc_auc / pr_auc as the headline metrics.


## 4b. VERIFY: augmentation summary is labelled as downstream, not classifier
The maintained summary sets `ddpm_is_classifier=False` and ablation labels. Confirm the labelling exists in the code that writes the summary.

In [6]:
agsrc = open('../training/latent_ddpm_augmentation.py').read()
assert '"ddpm_is_classifier": False' in agsrc
assert '"ablation_labels"' in agsrc
print('OK: DDPM rows are labelled downstream-classifier-after-augmentation')

OK: DDPM rows are labelled downstream-classifier-after-augmentation


## 4c. VERIFY (follow-up): best_result ranking criterion
`outputs/` is nearly empty locally. Search for any best_result artifact and, if present, confirm it records the ranking criterion; otherwise flag as a follow-up for a full run.

In [7]:
import glob, json
hits = glob.glob('../outputs/**/best_result*.json', recursive=True)
msg = ''
if not hits:
    msg = 'FOLLOW-UP: no best_result*.json locally (outputs empty). On a full run, confirm it records the ranking criterion / selection metric.'
    print(msg)
else:
    obj = json.load(open(hits[0]))
    keys = ' '.join(obj.keys()).lower()
    msg = 'has ranking criterion' if any(k in keys for k in ['metric','ranked','selected','criterion','objective']) else 'FOLLOW-UP: add ranking-criterion field'
    print(hits[0], '->', msg)
assert 'FOLLOW-UP' in msg or 'has ranking criterion' in msg

FOLLOW-UP: no best_result*.json locally (outputs empty). On a full run, confirm it records the ranking criterion / selection metric.


## 5. Deprecated layout: legacy code quarantined

In [8]:
from pathlib import Path
assert not Path('../data/data_247.pt').exists()
assert Path('../deprecated/data_247.pt').exists()
assert Path('../deprecated/train_pipeline.py').exists()
print('OK: legacy DDPM-as-classifier code + data_247 in deprecated/')

OK: legacy DDPM-as-classifier code + data_247 in deprecated/


## 6. Data validation on the canonical file

In [9]:
from review_fixes_2026_07.data_validation import validate_hetero_graph
data = torch.load('../data_ln_pc_ihc_g.pt', map_location='cpu', weights_only=False)
r = validate_hetero_graph(data)
print(r.info)
assert r.ok, r.problems
assert r.info['clinical_dim'] == 22 and r.info['class_counts'] == {0:62,1:185}
print('OK: canonical graph valid')

{'num_patients': 247, 'clinical_dim': 22, 'class_counts': {0: 62, 1: 185}, 'pathology_mask_true': 105, 'radiology_mask_true': 187, 'num_lesions': 333}
OK: canonical graph valid


## 7. Raw-data audit (auditable vs opaque)

In [10]:
from review_fixes_2026_07.raw_data_audit import run_raw_data_audit
rep = run_raw_data_audit('../data_ln_pc_ihc_g.pt', {
  'clinical_unscaled': '../data/x_clinical_unscaled_ln_pc_ihc_g.csv',
  'glcm': '../data/glcm_features.csv',
  'radiology': '../data/radiology_features.csv',
  'clinical_tmb': '../data/clinical_features_tmb.csv'})
print('AUDITABLE:', rep.auditable.get('clinical_unscaled_rows'), 'clinical rows vs',
      rep.auditable['graph_num_patients'], 'patients')
print('OPAQUE items:', len(rep.opaque))
assert rep.auditable['clinical_unscaled_rows'] == rep.auditable['graph_num_patients']
print('OK: per-patient clinical CSV aligns 1:1 with the graph')

AUDITABLE: 247 clinical rows vs 247 patients
OPAQUE items: 3
OK: per-patient clinical CSV aligns 1:1 with the graph


## 8. Bootstrap CIs on pooled out-of-fold predictions

In [11]:
from review_fixes_2026_07.bootstrap_ci import bootstrap_metric_cis
rng = np.random.default_rng(0)
# stand-in pooled OOF arrays; on a real run use kfold_evaluate_gvae_classifier's
# oof_arrays['y_true'], oof_arrays['head_probs'].
y = rng.integers(0,2,80); s = np.clip(0.5 + 0.3*(y-0.5) + 0.2*rng.standard_normal(80),0,1)
res = bootstrap_metric_cis(y, s, n_boot=500, seed=1)
for k,v in res.items(): print(f"{k}: {v['point']:.3f} [{v['ci_low']:.3f}, {v['ci_high']:.3f}]")
assert all(v['ci_low'] <= v['point'] <= v['ci_high'] for v in res.values())
print('OK: CIs bracket point estimates')
# TO SCALE UP: feed real pooled OOF (labels, head_probs) from a full run;
# raise n_boot to 2000.

roc_auc: 0.811 [0.713, 0.888]
pr_auc: 0.871 [0.780, 0.932]
balanced_accuracy: 0.704 [0.594, 0.792]
f1: 0.753 [0.636, 0.834]
OK: CIs bracket point estimates


## 9. Seed sweep aggregation (smoke: fake trainer)

In [12]:
from review_fixes_2026_07.seed_sweep import run_seed_sweep
def _fake(data, mc, tc): return {'mean_roc_auc': 0.55 + 0.03*tc['random_seed']}, None, None
sw = run_seed_sweep(None, {}, {'random_seed':0}, seeds=(0,1,2), train_fn=_fake)
print(sw['aggregate'])
assert sw['aggregate']['mean_roc_auc']['n_seeds'] == 3
print('OK: seed sweep aggregates across-seed distribution')
# TO SCALE UP: drop train_fn to use the real kfold_train_gvae, pass the
# canonical data + your model_config/train_config (shrink n_splits/epochs
# first), and use seeds=(0,1,2,3,4).

{'mean_roc_auc': {'mean': 0.5800000000000001, 'std': 0.024494897427831803, 'min': 0.55, 'max': 0.6100000000000001, 'n_seeds': 3, 'values': [0.55, 0.5800000000000001, 0.6100000000000001]}}
OK: seed sweep aggregates across-seed distribution


## 10. KNOWN FOLLOW-UP (not fixed this round)
PROJECT_REVIEW §15.5: the `filter_quantile=0.95` synthetic filter kept **zero** samples in filtered branches on the latest run. Diagnosed but not retuned — retuning needs a full run. When scaling up, sweep the quantile (e.g. 0.80-0.95) and confirm filtered branches retain samples before trusting filtered-ablation numbers.

In [13]:
follow_up_msg = 'Follow-up recorded: retune filter_quantile on a full run.'
print(follow_up_msg)
assert 'retune filter_quantile' in follow_up_msg

Follow-up recorded: retune filter_quantile on a full run.
